# 03 — Capacity Aggregation

Purpose:
Practice capacity aggregation SQL using PostgreSQL telemetry data.

This notebook covers:
- PostgreSQL connection
- smoke test
- helper function to run SQL
- table inspection helper
- DATE_TRUNC time buckets
- AVG / MAX / SUM
- P95 latency with PERCENTILE_CONT
- service-level capacity rollups
- daily and hourly telemetry summaries
- interview explanations


## Cell 2 — Install/import dependencies


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")


## Cell 3 — Connection settings


In [ ]:
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

password_encoded = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{password_encoded}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")

# If this notebook runs directly on Windows instead of inside a container,
# change DB_HOST to "localhost".


## Cell 4 — Smoke test connection


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, now();"))
    row = result.fetchone()

row


## Cell 5 — Helper function to run SQL


In [ ]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)


## Cell 6 — Helper function to inspect one table safely


In [ ]:
def inspect_table_safe(table_name: str) -> None:
    """
    Safely inspect a table without changing data.
    Shows column metadata, row count, and a small preview.
    """
    metadata_sql = f"""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position;
    """

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    preview_sql = f"""
    SELECT *
    FROM {table_name}
    LIMIT 10;
    """

    print(f"Column metadata for public.{table_name}")
    display(run_sql(metadata_sql))

    print(f"Row count for public.{table_name}")
    display(run_sql(count_sql))

    print(f"Preview rows from public.{table_name}")
    display(run_sql(preview_sql))


# 03 — Capacity Aggregation Practice


## 03.1 Verify tables used in this notebook

This notebook uses:
- `telemetry_samples` as the metric fact table
- `services` as the service lookup table
- `hosts` as the host lookup table when needed


In [ ]:
sql = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN ('telemetry_samples', 'services', 'hosts')
ORDER BY table_name;
"""

run_sql(sql)


## 03.2 Inspect telemetry_samples

`telemetry_samples` contains sampled capacity metrics:
CPU, memory, latency, request rate, error rate, allocated resources,
actual resources, cost, and JSONB tags.


In [ ]:
inspect_table_safe("telemetry_samples")


## 03.3 Service-level average CPU and memory

This query groups many telemetry rows into one capacity summary row per service.
AVG shows normal or typical utilization.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY avg_cpu_pct DESC;
"""

run_sql(sql)


## 03.4 Service-level AVG and MAX capacity

AVG shows typical utilization.
MAX shows the worst observed spike.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY max_cpu_pct DESC;
"""

run_sql(sql)


## 03.5 P95 latency by service

`p95_latency_ms` is already a sampled P95 latency value.
This query calculates the P95 of those sampled P95 values per service.
This is an operational rollup, not true raw request-level P95.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.p95_latency_ms)::NUMERIC,
        2
    ) AS p95_of_sampled_p95_latency_ms
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY p95_of_sampled_p95_latency_ms DESC;
"""

run_sql(sql)


## 03.6 Daily service rollup

`DATE_TRUNC('day', sampled_at)` buckets timestamped telemetry into daily windows.
This creates one row per day and service.


In [ ]:
sql = """
SELECT
    DATE_TRUNC('day', t.sampled_at) AS sample_day,
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(AVG(t.requests_per_min), 0) AS avg_requests_per_min,
    ROUND(AVG(t.error_rate_pct), 3) AS avg_error_rate_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('day', t.sampled_at),
    s.service_name
ORDER BY
    sample_day,
    s.service_name;
"""

run_sql(sql)


## 03.7 Hourly service rollup

Hourly buckets are useful when telemetry arrives every few minutes and we want operational trends.


In [ ]:
sql = """
SELECT
    DATE_TRUNC('hour', t.sampled_at) AS sample_hour,
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct,
    ROUND(AVG(t.requests_per_min), 0) AS avg_requests_per_min
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('hour', t.sampled_at),
    s.service_name
ORDER BY
    sample_hour,
    s.service_name;
"""

run_sql(sql)


## 03.8 Hourly AVG, MAX, and P95 for CPU and memory

For CPU and memory:
- AVG = normal utilization
- MAX = worst observed spike
- P95 = sustained high pressure while reducing one-off noise


In [ ]:
sql = """
SELECT
    DATE_TRUNC('hour', t.sampled_at) AS sample_hour,
    s.service_name,

    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(t.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.cpu_utilization_pct)::NUMERIC,
        2
    ) AS p95_cpu_pct,

    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(t.memory_utilization_pct), 2) AS max_memory_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY t.memory_utilization_pct)::NUMERIC,
        2
    ) AS p95_memory_pct

FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY
    DATE_TRUNC('hour', t.sampled_at),
    s.service_name
ORDER BY
    sample_hour,
    p95_cpu_pct DESC;
"""

run_sql(sql)


## 03.9 Capacity waste / over-allocation check

This compares allocated resources to actual usage.
It helps identify possible over-provisioning.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.allocated_cpu_cores,
    t.actual_cpu_cores,
    ROUND(t.allocated_cpu_cores - t.actual_cpu_cores, 2) AS unused_cpu_cores,
    t.allocated_memory_gb,
    t.actual_memory_gb,
    ROUND(t.allocated_memory_gb - t.actual_memory_gb, 2) AS unused_memory_gb,
    t.cloud_cost_usd
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
WHERE t.allocated_cpu_cores > t.actual_cpu_cores
   OR t.allocated_memory_gb > t.actual_memory_gb
ORDER BY
    t.cloud_cost_usd DESC,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 03.10 Threshold-style capacity risk

This finds samples where utilization, latency, error rate, or forecast values cross risk thresholds.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms,
    t.error_rate_pct,
    t.forecast_cpu_pct,
    t.forecast_memory_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
WHERE t.cpu_utilization_pct >= 85
   OR t.memory_utilization_pct >= 85
   OR t.p95_latency_ms >= 500
   OR t.error_rate_pct >= 2
   OR t.forecast_cpu_pct >= 85
   OR t.forecast_memory_pct >= 85
ORDER BY
    t.sampled_at,
    s.service_name,
    t.host_id;
"""

run_sql(sql)


## 03.11 Cost rollup by service

If `cloud_cost_usd` is incremental per sample, `SUM` gives total cost over the period.
If it is a point-in-time snapshot, `AVG` or `MAX` may be more appropriate.
For this practice, treat it as incremental sample cost.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(SUM(t.cloud_cost_usd), 2) AS total_cloud_cost_usd,
    ROUND(AVG(t.cloud_cost_usd), 4) AS avg_sample_cloud_cost_usd,
    COUNT(*) AS sample_count
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY total_cloud_cost_usd DESC;
"""

run_sql(sql)


## 03.12 Final interview explanation

Capacity aggregation turns noisy telemetry samples into useful operational summaries. I use DATE_TRUNC to bucket timestamps into hours or days. For CPU and memory, AVG shows typical utilization, MAX shows peak pressure, and P95 shows sustained high utilization while reducing one-off noise. For latency, P95 is especially useful because user experience is often affected by tail latency. If the source column already stores 5-minute P95 latency, then the hourly calculation is P95 of those sampled P95 values unless raw request-level events are available.
